# TTM FC01 — Owner A/B/C one-shot T4 FP32
**User explicitly authorized execution on 13/11/2026.** One logical run of A/B/C using frozen FC01 V2 sources, FP32 and a free Tesla T4. No source hyperparameter changes, no TEST/Q1Q2 access, no rerun after a failure.

**Precondition:** Upload *only three missing frozen Python dependencies* into existing private Google Drive `input/` alongside the seven previously pinned files: `fc01_build_private_corpus.py`, `fc01_integrated_static_qa.py`, `fc01_pinned_bpe.py` (they are staged via Windows PS helper). All three dependencies have now been uploaded directly to private Drive and SHA-256 verified. This notebook verifies the already issued GO lock before running. No need to share private corpus publicly.

**One-shot:** Read last owner rights+free-T4 Colab preflight receipt; verify input SHA and P02 Git blob; run source-only audit; verify the pre-issued **private** `FC01_OWNER_GO.json` by SHA-256 (created after explicit owner chat authorization), then train A → B → C, evaluate EVAL64, and produce secondary blind96 packet. Training operations run as separate child processes with `CUBLAS_WORKSPACE_CONFIG=:4096:8`. Frozen runner is not edited.

**Important:** No *automatic retry* or re-seeding. On any exception, the notebook writes a terminal HOLD receipt if the science output folder exists. A real step64 resume was not demonstrated by step0 preflight; fail closed on resume or CUDA divergence. The source-only GO from the prior operation is readiness, not evidence this training already ran.

To launch: Open this notebook in an actual **Colab Free Tesla T4** session and run its one code cell **once**. Runtime disconnects, empty final reports and partial checkpoints are not science PASS.


In [ ]:
# TTM FC01 V2 — ONE logical A/B/C science execution, no rerun/reseed/retraining on error.
# Owner has explicitly authorized this operation in ChatGPT on 2026-11-13.
# Execute this cell once on the user's Colab FREE Tesla T4 runtime.
# Do not run on Windows / paid Colab / another GPU model.

import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # before any runner CUDA process
os.environ["PYTHONHASHSEED"] = "0"
import sys, json, hashlib, time, datetime, subprocess, pathlib, traceback, shutil

BASE = pathlib.Path("/content/drive/MyDrive/TTM/ttm-small-pilot/ttm_small4m_fc01_fact_conditioning_20261113")
INPUT = BASE / "input"
RECEIPTS = BASE / "preexec_receipts"
OUT = BASE / "FC01_ABC_ONE_SHOT_V2_20261113"  # unique frozen destination
RUNNER = INPUT / "fc01_train_eval_once.py"
GO = INPUT / "FC01_OWNER_GO.json"
GO_SHA_PIN = "8b274e3bc976e73ad25a689e8044dda2fcb98766a8d2cdd890cb08d6f582c091"
ARMS = ("A_BASE_MLE","B_CF_MLE","C_CF_CONTRAST")
SOURCE_HASHES = {
    "FC01_OWNER_PRIVATE_DATA.json":"becc0408880aaed8bd5eb6fdf3997e62d5131c187a48331a03ae37d41aaa1aa3",
    "T4_PRIVATE_TRAIN_VAL_WINDOWS.npz":"184f68b4c0c800ef3e98362c2e20e1cd35baa269e926489a475664526ecd9913",
    "T4_PRIVATE_PACK_MANIFEST.json":"ef2e6e67d49b1d5a921b1051df329f15574f1c10cece0ce49468acb53322d650",
    "CF1_STEP1024_PINNED.pt":"cb985751795c8328ad65acf7ca4c1033ce35182c1bfbdb8c38550de629cb271b",
    "T1_BPE2048_REAL_TRAIN_MANIFEST.json":"cba18e60cee3d08a353ee6d4a451017289e45af7b4b1179c87171b9b78d7dfdc",
    "fc01_train_eval_once.py":"b786a4468ba104786df7683da9b1cf19d2334d9cc203dd19c3e7d6f10e2e93be",
    "fc01_build_private_corpus.py":"a9978a1961a2d062e5229ad91593eddb01f7fcfab00c4dfb55cc3df316053c98",
    "fc01_integrated_static_qa.py":"f4f205ed4be0021faf99b3e73f4c0fb80ef4d1ce095ee5985507ca3f1e33cc3e",
    "fc01_pinned_bpe.py":"d90ddbcdbea8450d7605cf7e65936efd032344b8d7793c2272d4d0870cfbff01",
}
P02_BLOB="9762343e06cf800b50574c2871ae1ed58537fb18"
PREEXEC_JSON=RECEIPTS/"FC01_T4_FP32_DRIVE_ZERO_STEP_READINESS_20261113T130706Z.json"
PREEXEC_SHA="f7d833493175a61e0e8cb13ec6805957b659e909d2d4fdd72d3d25796f4a23ac"
PREEXEC_SNAPSHOT=RECEIPTS/"FC01_ZERO_STEP_DURABILITY_20261113T130706Z.pt"
PREEXEC_SNAPSHOT_SHA="2a31952b268ddbbafff4f32500e866d5341a3ea576a6219dbe830b5304cec1fd"
GO_OPERATION="TTM_SMALL4M_FC01_OWNER_SINGLE_GO_LOCK_AND_ABC_ONE_SHOT_T4_FP32_EXECUTION"
TOTAL_LIMIT_SECONDS=10800
ARM_LIMIT_SECONDS=3600

def utc():
    return datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
def sha(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for b in iter(lambda:f.read(1<<20),b""):h.update(b)
    return h.hexdigest()
def git_blob(p):
    b=p.read_bytes()
    return hashlib.sha1(b"blob "+str(len(b)).encode("ascii")+b"\0"+b).hexdigest()
def canonical(obj):
    return (json.dumps(obj,sort_keys=True,separators=(",",":"),ensure_ascii=True,allow_nan=False)+"\n").encode("ascii")
def write_new(path, content):
    with path.open("xb") as f:
        f.write(content)
        f.flush()
        os.fsync(f.fileno())
def stop(s):
    raise RuntimeError(s)

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
if not (INPUT.is_dir() and RECEIPTS.is_dir() and BASE.is_dir()):
    stop("FC01_DRIVE_INPUT_OR_RECEIPTS_MISSING")
if OUT.exists():
    stop("FC01_ONE_SHOT_ALREADY_EXECUTED_OR_PARTIAL_NO_DUPLICATE_OR_RESTART")
if not GO.is_file() or sha(GO)!=GO_SHA_PIN:
    stop("FC01_EXACT_OWNER_GO_LOCK_MISSING_OR_MODIFIED")
if sys.version_info.major != 3:
    stop("FC01_PYTHON3_REQUIRED")

# Exact source, model and historical owner authorization receipt all verified before GO.
for name, exp in SOURCE_HASHES.items():
    p=INPUT/name
    if not p.is_file() or p.is_symlink():
        stop("FC01_MISSING_EXACT_DEPENDENCY_OR_INPUT:"+name)
    if sha(p)!=exp:
        stop("FC01_FROZEN_SHA_MISMATCH_NO_TRAIN:"+name)
if not (INPUT/"p02_exact.py").is_file() or git_blob(INPUT/"p02_exact.py") != P02_BLOB:
    stop("FC01_P02_GIT_BLOB_DIFFERENT")
if sha(PREEXEC_JSON)!=PREEXEC_SHA or sha(PREEXEC_SNAPSHOT)!=PREEXEC_SNAPSHOT_SHA:
    stop("FC01_ZERO_STEP_RECEIPT_OR_DRIVE_SNAPSHOT_SHA_MISMATCH")
previous=json.loads(PREEXEC_JSON.read_text(encoding="utf-8"))
if (previous.get("status")!="PASS_COLAB_ZERO_STEP_SCOPED_ONLY_NOT_SCIENCE_GO"
    or previous.get("owner_rights_acknowledged") is not True
    or previous.get("owner_free_t4_cost_cap_acknowledged") is not True
    or previous.get("checkpoint_roundtrip")!="PASS_ZERO_STEP_MODEL_OPTIMIZER_RNG_CURSOR_DRIVE"
    or previous.get("science_optimizer_steps")!=0):
    stop("FC01_PREVIOUS_ZERO_STEP_OWNER_RIGHTS_OR_RUNTIME_NOT_ACCEPTED")
if previous.get("source_hashes") != {
    name:SOURCE_HASHES[name] for name in (
    "CF1_STEP1024_PINNED.pt","FC01_OWNER_PRIVATE_DATA.json",
    "T1_BPE2048_REAL_TRAIN_MANIFEST.json","T4_PRIVATE_PACK_MANIFEST.json",
    "T4_PRIVATE_TRAIN_VAL_WINDOWS.npz","fc01_train_eval_once.py"
    )}:
    stop("FC01_PREVIOUS_PREEXEC_SOURCE_HASHES_DIFFERENT")
if previous.get("p02_git_blob") != P02_BLOB:
    stop("FC01_PREVIOUS_PREEXEC_P02_BLOB_DIFFERENT")
if previous.get("preflight_snapshot_sha256") != PREEXEC_SNAPSHOT_SHA:
    stop("FC01_PREVIOUS_PREEXEC_SNAPSHOT_NOT_PINNED")
import torch
if not torch.cuda.is_available() or "T4" not in torch.cuda.get_device_name(0).upper():
    stop("FC01_FREE_T4_DEVICE_NOT_AVAILABLE")
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
print("FC01_FROZEN_INPUTS=PASS_10_OF_10 (9 SHA256 + P02 Git blob)")
print("FC01_OWNER_APPROVAL=CHAT_EXPLICIT_AND_PREEXEC_RIGHTS_ATTESTED")
print("FC01_T4=PASS FP32=REQUIRED ONE_SHOT=TRUE TRAINING_NOT_STARTED")

# Re-run original pinned exact source-only audit once in a fresh subprocess.
ENV=os.environ.copy()
ENV["CUBLAS_WORKSPACE_CONFIG"]=":4096:8"
ENV["PYTHONHASHSEED"]="0"
ENV["CUDA_VISIBLE_DEVICES"]="0"
def invoke(cmd, logfile, timeout):
    with logfile.open("xb") as f:
        p=subprocess.run([sys.executable,str(RUNNER)]+list(cmd),
                         cwd=str(INPUT),env=ENV,stdout=f,stderr=subprocess.STDOUT,
                         timeout=max(1,int(timeout)),check=False)
        f.flush()
        os.fsync(f.fileno())
    print("FC01_PHASE="+cmd[1]+" EXIT="+str(p.returncode)+" LOG_SHA256="+sha(logfile),flush=True)
    if p.returncode:
        stop("FC01_RUNNER_NONZERO_NO_RETRY:"+str(p.returncode)+":LOG="+str(logfile))
    return logfile.name
auditlog=RECEIPTS/"FC01_SCIENCE_ONE_SHOT_LAST_PREEXEC_AUDIT_20261113.log"
invoke(["--mode","audit","--inputs-dir",str(INPUT)],auditlog,300)
print("FC01_EXACT_AUDIT_RECONFIRMED=PASS",flush=True)

# Owner GO was emitted only once to the private Drive input from the user's explicit approval.
# It is exact-SHA pinned and is NOT a cryptographic human signature.
go=json.loads(GO.read_text(encoding="utf-8"))
if (go.get("authorization")!="FC01_ALL_THREE_ARMS_ONE_SHOT_GO"
    or go.get("all_arms")!=list(ARMS)
    or go.get("source_rights_accepted") is not True
    or go.get("owner_free_t4_approved") is not True
    or go.get("single_use") is not True):
    stop("FC01_OWNER_GO_CONTENT_OR_ONE_SHOT_POLICY_MISMATCH")
go_sha=GO_SHA_PIN
print("FC01_OWNER_GO_PIN_VERIFIED="+go_sha,flush=True)
OUT.mkdir(parents=False,exist_ok=False)
metadata={
   "schema":"ttm.fc01.abc.one_shot.operator_state.v1",
   "operation":GO_OPERATION,
   "science_go_sha256":go_sha,
   "owner_rights_source":"EXPLICIT_USER_2026-11-13_AND_COLAB_PREEXEC",
   "source_sha256":SOURCE_HASHES,
   "p02_git_blob":P02_BLOB,
   "arms":list(ARMS),
   "start_utc":utc(),
   "max_gpu_wall_seconds":TOTAL_LIMIT_SECONDS,
   "result":"RUNNING",
   "completed_arms":[],
   "new_optimizer_steps_target":768,
   "new_optimizer_steps_observed":"NOT_YET_ADJUDICATED",
   "science_execution_started":False,
   "attempted_arm":None,
   "test_q1q2":"SEALED",
   "d1_confirmatory":"HOLD",
   "restart_policy":"ONE_SHOT_NO_AUTOMATIC_RETRY_OR_RESUBMIT",
}
write_new(OUT/"FC01_OWNER_GO_PIN_AND_SCOPE.json",canonical(metadata))
start=time.monotonic()
terminal=None
completed=[]
final_sha={}
try:
    print("FC01_GO_LOCK_SHA256="+go_sha,flush=True)
    print("FC01_SCIENCE_ABC_DISPATCH=STARTED_ONE_LOGICAL_RUN",flush=True)
    for arm in ARMS:
        remaining=TOTAL_LIMIT_SECONDS-(time.monotonic()-start)
        if remaining<=0:stop("FC01_TOTAL_GPU_BUDGET_EXHAUSTED")
        metadata["attempted_arm"]=arm
        metadata["science_execution_started"]=True
        print("FC01_ARM_START="+arm,flush=True)
        logfile=OUT/("FC01_PHASE_"+arm+".log")
        invoke(["--mode","train","--inputs-dir",str(INPUT),
                "--outputs-dir",str(OUT),"--arm",arm,"--go-sha256",go_sha],
                logfile,min(ARM_LIMIT_SECONDS,remaining))
        final=OUT/arm/("FC01_"+arm+"_STEP256.pt")
        if not final.is_file():stop("FC01_ARM_FINAL_CHECKPOINT_MISSING:"+arm)
        final_sha[arm]=sha(final)
        completed.append(arm)
        print("FC01_ARM_FINAL_SHA256="+arm+":"+final_sha[arm],flush=True)
    remaining=TOTAL_LIMIT_SECONDS-(time.monotonic()-start)
    if remaining<=0:stop("FC01_TOTAL_GPU_BUDGET_EXHAUSTED_BEFORE_EVAL")
    invoke(["--mode","evaluate","--inputs-dir",str(INPUT),
            "--outputs-dir",str(OUT),"--go-sha256",go_sha],
            OUT/"FC01_PHASE_EVAL64.log",remaining)
    if not (OUT/"FC01_EVAL64_PUBLIC_SAFE_RECEIPT.json").is_file():
        stop("FC01_EVAL64_PUBLIC_SAFE_RECEIPT_MISSING")
    remaining=TOTAL_LIMIT_SECONDS-(time.monotonic()-start)
    if remaining<=0:stop("FC01_TOTAL_GPU_BUDGET_EXHAUSTED_BEFORE_SECONDARY")
    invoke(["--mode","secondary","--inputs-dir",str(INPUT),
            "--outputs-dir",str(OUT),"--go-sha256",go_sha],
            OUT/"FC01_PHASE_SECONDARY.log",remaining)
    for name in ("FC01_SECONDARY_BLIND96_PRIVATE.json",
                 "FC01_SECONDARY_OWNER_MAPPING_PRIVATE.json",
                 "FC01_SECONDARY_PUBLIC_SAFE_RECEIPT.json"):
        if not (OUT/name).is_file():stop("FC01_MISSING_SECONDARY_"+name)
    terminal="COMPLETED_SOURCE_LOCKED_PRIMARY_AND_UNSCORED_SECONDARY"
except BaseException as exc:
    terminal="HOLD_TERMINAL_NO_AUTO_RETRY"
    metadata["failure"]=type(exc).__name__+":"+str(exc)
    print("FC01_ONE_SHOT_TERMINAL_HOLD="+metadata["failure"],flush=True)
finally:
    metadata["result"]=terminal or "HOLD_TERMINAL_UNKNOWN"
    metadata["completed_arms"]=completed
    metadata["final_step256_sha256"]=final_sha
    metadata["elapsed_wall_seconds"]=round(time.monotonic()-start,3)
    metadata["completed_utc"]=utc()
    metadata["science_executed"]=bool(metadata.get("science_execution_started",False))
    metadata["new_optimizer_steps_observed"]=(768 if len(completed)==3 else "UNDETERMINED_IF_TRAIN_INTERRUPTED")
    metadata["test_q1q2"]="SEALED"
    metadata["d1_confirmatory"]="HOLD"
    if (OUT/"FC01_EVAL64_PUBLIC_SAFE_RECEIPT.json").is_file():
        metadata["eval64_public_safe_sha256"]=sha(OUT/"FC01_EVAL64_PUBLIC_SAFE_RECEIPT.json")
    if (OUT/"FC01_SECONDARY_PUBLIC_SAFE_RECEIPT.json").is_file():
        metadata["secondary_public_safe_sha256"]=sha(OUT/"FC01_SECONDARY_PUBLIC_SAFE_RECEIPT.json")
    receipt=OUT/"FC01_ABC_ONE_SHOT_TERMINAL_RECEIPT.json"
    write_new(receipt,canonical(metadata))
    print("FC01_ONE_SHOT_TERMINAL_STATUS="+metadata["result"],flush=True)
    print("FC01_ONE_SHOT_TERMINAL_RECEIPT="+str(receipt),flush=True)
    print("FC01_ONE_SHOT_TERMINAL_RECEIPT_SHA256="+sha(receipt),flush=True)
if terminal!="COMPLETED_SOURCE_LOCKED_PRIMARY_AND_UNSCORED_SECONDARY":
    stop("FC01_ABC_ONE_SHOT_HOLD_NO_AUTO_RETRY")
print("FC01_ABC_ONE_SHOT=COMPLETE_PENDING_INDEPENDENT_SCIENTIFIC_ADJUDICATION")
print("FC01_SCIENCE_OWNER_GO_SHA256="+go_sha)
print("FC01_D1=HOLD FC01_TEST_Q1Q2=SEALED FC01_NO_AUTO_RETRY=TRUE")
